# How to find companies with going-concern warnings

Build a list of companies whose filings say there is substantial doubt they can continue as a going concern, from 10-Qs and 10-Ks, with tickers, in Python.

Read it with charts at https://www.quantisnow.com/sec-api/guides/find-going-concern-companies. More notebooks: https://github.com/quantisnow/sec-edgar-notebooks. Set `QUANTISNOW_API_KEY` before running; make a key at https://www.quantisnow.com/sec-api/keys.

Under US accounting rules, management has to assess every quarter whether the company can meet its obligations for the next twelve months. When it can't rule that out, the filing has to say there is substantial doubt about its ability to continue as a going concern. In a 10-K the auditor can add the same warning to its opinion. It is one of the strongest single signals of distress in public filings, and the wording is close to fixed, which makes it a good full-text search.

## The query

Two phrases joined with AND, so the filing contains both. Searching quarterly reports catches a warning up to three months before the next annual report would.

### Which companies said there is substantial doubt they can continue as a going concern?

Auditors and management have to say so when a company may not last another year. Credit desks and short sellers read this list every quarter.

```shell
curl -X POST https://www.quantisnow.com/api/v1/full-text \
  -H "Authorization: Bearer $QUANTISNOW_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
  "query": "\"substantial doubt\" AND \"going concern\"",
  "formTypes": ["10-Q", "10-Q/A"],
  "sort": "filedAt:desc",
  "size": 5,
  "facets": true
}'
```

Response (real, first 2 of 684):

```json
{
  "total": {
    "value": 684,
    "relation": "eq"
  },
  "took": 402,
  "data": [
    {
      "id": "0001493152-26-053834",
      "kind": "filing",
      "source": "SEC",
      "accessionNo": "0001493152-26-053834",
      "formType": "10-Q",
      "filedAt": "2026-11-23T17:27:40-05:00",
      "cik": "0001392694",
      "companyName": "SurgePays Inc.",
      "ticker": "SURG",
      "tickers": [
        "SURG"
      ],
      "items": [],
      "description": "SEC Form 10-Q filed by SurgePays Inc.",
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/1392694/000149315226053834/0001493152-26-053834-index.htm",
      "quantisnowUrl": "https://www.quantisnow.com/insight/sec-form-10-q-filed-by-surgepays-inc-6804422",
      "score": null,
      "highlights": [
        "<em>doubt</em> about the Company's ability to continue as a <em>going concern</em> within one year after the date these consolidated financial statements are issued. 9 | Management has evaluated the significance of these conditions and has",
        "developed plans intended to mitigate the <em>substantial doubt</em>.",
        "Accordingly, management has concluded that its plans do not alleviate the <em>substantial doubt</em> about the Company's ability to continue as a <em>going concern</em>."
      ]
    },
    {
      "id": "0001477932-26-007251",
      "kind": "filing",
      "source": "SEC",
      "accessionNo": "0001477932-26-007251",
      "formType": "10-Q",
      "filedAt": "2026-11-23T17:27:21-05:00",
      "cik": "0001425627",
      "companyName": "SOBR Safe Inc.",
      "ticker": "SOBR",
      "tickers": [
        "SOBR"
      ],
      "items": [],
      "description": "SEC Form 10-Q filed by SOBR Safe Inc.",
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/1425627/000147793226007251/0001477932-26-007251-index.htm",
      "quantisnowUrl": "https://www.quantisnow.com/insight/sec-form-10-q-filed-by-sobr-safe-inc-6804398",
      "score": null,
      "highlights": [
        "Accordingly, <em>substantial doubt</em> about the Company's ability to continue as a <em>going concern</em> has not been alleviated as of the date these financial statements are issued. 15 Table of Contents NOTE 3.",
        "Our current cash is not sufficient to fund our operations for the twelve months following the issuance of our financial statements, and there is <em>substantial doubt</em> about our ability to continue as a <em>going concern</em>.",
        "These conditions raise <em>substantial doubt</em> about the Company's ability to continue as a <em>going concern</em> for the twelve months following the date these financial statements are issued."
      ]
    }
  ],
  "facets": {
    "formType": [
      {
        "value": "10-Q",
        "count": 679
      },
      {
        "value": "10-Q/A",
        "count": 5
      }
    ],
    "month": [
      {
        "value": "2026-11",
        "count": 678
      },
      {
        "value": "2026-10",
        "count": 6
      }
    ],
    "companyName": [
      {
        "value": "Optimum Communications Inc.",
        "count": 3
      },
      {
        "value": "Apnimed Inc.",
        "count": 2
      },
      {
        "value": "Hepion Pharmaceuticals Inc.",
        "count": 2
      },
      {
        "value": "Spark I Acquisition Corp.",
        "count": 2
      },
      {
        "value": "1RT Acquisition Corp.",
        "count": 1
      },
      {
        "value": "22nd Century Group Inc.",
        "count": 1
      },
      {
        "value": "60 Degrees Pharmaceuticals Inc.",
        "count": 1
      },
      {
        "value": "8x8 Inc",
        "count": 1
      },
      {
        "value": "A SPAC III Acquisition Corp.",
        "count": 1
      },
      {
        "value": "AA Mission Acquisition Corp. II",
        "count": 1
      }
    ],
    "source": [
      {
        "value": "SEC",
        "count": 684
      }
    ]
  }
}
```

## Turn the hits into a list of companies

Page through every match and keep one row per company. Count companies by `cik`: tickers change, and some filers, like blank-check companies, have none.

In [ ]:
import os, re, requests
import pandas as pd

URL = "https://www.quantisnow.com/api/v1/full-text"
KEY = os.environ["QUANTISNOW_API_KEY"]
QUERY = '"substantial doubt" AND "going concern"'

hits, start = [], 0
while True:
    page = requests.post(URL, headers={"Authorization": f"Bearer {KEY}"}, json={
        "query": QUERY, "formTypes": ["10-Q", "10-Q/A", "10-K", "10-K/A"],
        "sort": "filedAt:desc", "from": start, "size": 50,
    }).json()
    hits += page["data"]
    start += 50
    if start >= page["total"]["value"]:
        break

df = pd.DataFrame([{
    "cik": h["cik"], "ticker": h["ticker"], "company": h["companyName"],
    "form": h["formType"], "filed": h["filedAt"][:10],
    "passage": re.sub(r"</?em>", "", h["highlights"][0]) if h["highlights"] else "",
} for h in hits])
latest = df.sort_values("filed").groupby("cik").tail(1)
print(len(latest), "companies")
latest.to_csv("going_concern.csv", index=False)

## Remove the false positives

- Doubt resolved: "substantial doubt ... has been alleviated" or "is not raised". Add `NOT alleviated` if you only want open warnings, and accept that it also drops filings that describe plans to alleviate it.
- Blank-check companies: almost every SPAC carries the warning because its trust has a deadline. Drop names ending in "Acquisition Corp", or filers with no ticker.
- Risk factors that talk about a future warning, like "may raise substantial doubt". Requiring `"raise substantial doubt"` or `"substantial doubt exists"` keeps the present-tense ones.

## Add the balance sheet

A going-concern warning matters most next to how much cash is left. The [financial statements endpoint](https://www.quantisnow.com/sec-api/financial-statements) has `balanceSheet.cash` and `cashFlow.operatingCashFlow` for each quarter, keyed by the same ticker, so months of runway is one more request per company.